# NB1 — EMCAD-B0 baseline on CVC-ClinicDB (official EMCAD code + recipe)

**Purpose:** reference run for the EMCAD + QML study. Every later notebook (QAG on LGAG, quantum multi-scale fusion, quantum head aggregation) reuses this exact pipeline and changes ONE component.

| Item | Value |
|---|---|
| Model | EMCADNet, encoder `pvt_v2_b0` (ImageNet-pretrained), official `lib/` code |
| Split | EMCAD official ClinicDB folders (train/val/test) |
| Recipe | EMCAD `train_polyp.py`: 352 input, batch 8, 200 epochs, AdamW lr 5e-4 wd 1e-4, cosine (T_max=200, eta_min=1e-6), grad value-clip 0.5, multi-scale {0.75,1,1.25}, augmentation (rotate, flips) |
| Loss | structure loss (weighted BCE + weighted IoU) on p4, p3, p2, p1 and on their sum |
| Eval | official `test()`: last head (p1), resize to original, sigmoid, per-image min-max, pred ≥ 0.5, GT ≥ 0.2 |
| Selection | best val Dice; test evaluated ONCE on that checkpoint (= EMCAD's "test Dice at best val") |

**Two quirks of the official code, reproduced on purpose (flags in CONFIG):**
1. `MULTISCALE_MODE = "official"`: in `train_polyp.py` the resized batch overwrites `images`, so the rate-1.0 step trains at 256 px and the 1.25 step is upsampled from 256 px. `"fixed"` resizes from the original 352 batch each time.
2. Eval resizes to `(W, H)` instead of `(H, W)` (PIL `size` is W,H). The final test cell reports the official number AND the orientation-corrected one.

Copyright © 2021, The University of Texas at Austin. All rights reserved. EMCAD code is used under the UT Austin Research License (academic use; derivative works are shared with the licensor). Keep this notebook private.

In [ ]:
# ============================================================
# Cell 0 — Pin to one GPU (run before importing torch)
# ============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Pinned to physical GPU 0")

In [ ]:
# ============================================================
# Cell 1 — Imports, environment, pretrained PVTv2 weights
# ============================================================
import os, json, csv, random, time, glob, math, warnings
from functools import partial
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import timm
from timm.models.layers import DropPath, to_2tuple, trunc_normal_, trunc_normal_tf_
from timm.models.helpers import named_apply
try:
    from thop import profile as thop_profile
except ImportError:
    os.system("pip install -q thop")
    from thop import profile as thop_profile

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | timm {timm.__version__} | albumentations {A.__version__}")
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)}, "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)" if DEVICE.type == "cuda" else ""))
torch.backends.cudnn.benchmark = True

# PVTv2 ImageNet weights (official PVT release). If Internet is off, upload the
# .pth as a Kaggle dataset and set PRETRAINED_DIR in CONFIG to that folder.
PVT_URL = "https://github.com/whai362/PVT/releases/download/v2/pvt_v2_b0.pth"
DEFAULT_PRETRAINED_DIR = "/kaggle/working/pretrained_pth/pvt" if os.path.isdir("/kaggle/working") else "./pretrained_pth/pvt"
os.makedirs(DEFAULT_PRETRAINED_DIR, exist_ok=True)
_w = os.path.join(DEFAULT_PRETRAINED_DIR, "pvt_v2_b0.pth")
if not os.path.exists(_w):
    print("Downloading pvt_v2_b0.pth ...")
    torch.hub.download_url_to_file(PVT_URL, _w, progress=False)
print(f"pvt_v2_b0.pth: {os.path.getsize(_w)/1e6:.1f} MB  ({_w})")

In [ ]:
# ============================================================
# Cell 2 — CONFIG (edit this cell only)
# ============================================================
RUN_NUMBER = 1                  # bump for repeat runs (EMCAD reports the mean of 5)
SEED       = 42                 # same seed for every notebook in this study

EXPERIMENT = "emcad_b0_baseline"
VARIANT    = "lgag"             # gate in use: original EMCAD LGAG (baseline)

CLINICDB_ROOT  = "/kaggle/input/datasets/moon1570/clinicdb/ClinicDB"   # train/ val/ test/ -> images/ masks/
PRETRAINED_DIR = DEFAULT_PRETRAINED_DIR

# Reproduce-the-official-code switches (see the markdown cell at the top)
MULTISCALE_MODE = "official"    # "official" (as in train_polyp.py) | "fixed"
USE_AMP         = False         # EMCAD trains in fp32. True = faster, small deviation.

RESUME          = False
RESUME_CKPT     = None          # path to a last.pth from a previous Kaggle version (Add Input)

KAGGLE_WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./outputs_local"
SLUG = f"{EXPERIMENT}_clinicdb_run{RUN_NUMBER}"

CONFIG = {
    "experiment": EXPERIMENT, "variant": VARIANT, "run_number": RUN_NUMBER, "seed": SEED, "slug": SLUG,
    # data
    "dataset": "CVC-ClinicDB", "split": "EMCAD official", "clinicdb_root": CLINICDB_ROOT,
    # model (train_polyp.py defaults)
    "encoder": "pvt_v2_b0", "pretrained_dir": PRETRAINED_DIR,
    "kernel_sizes": [1, 3, 5], "expansion_factor": 2, "lgag_ks": 3,
    "activation_mscb": "relu6", "dw_parallel": True, "add": True,
    # recipe (train_polyp.py defaults)
    "img_size": 352, "batch_size": 8, "test_batch_size": 8, "epochs": 200,
    "lr": 5e-4, "weight_decay": 1e-4, "cosine_eta_min": 1e-6, "clip": 0.5,
    "size_rates": [0.75, 1, 1.25], "augmentation": True, "color_image": True,
    "multiscale_mode": MULTISCALE_MODE, "use_amp": USE_AMP,
    "loss": "structure_loss x (p4,p3,p2,p1,sum)", "eval_head": "p1 (ress[-1])",
    "num_workers": 4,
    # outputs
    "output_dir": os.path.join(KAGGLE_WORKING, "outputs", SLUG),
    "ckpt_dir":   os.path.join(KAGGLE_WORKING, "checkpoints", SLUG),
}
for d in (CONFIG["output_dir"], CONFIG["ckpt_dir"]):
    os.makedirs(d, exist_ok=True)
with open(os.path.join(CONFIG["output_dir"], "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(SEED)

print(f"=== {SLUG} ===")
for k in ("encoder", "img_size", "batch_size", "epochs", "lr", "multiscale_mode", "use_amp"):
    print(f"  {k:<16}: {CONFIG[k]}")
for sp in ("train", "val", "test"):
    for sub in ("images", "masks"):
        p = os.path.join(CLINICDB_ROOT, sp, sub)
        print(f"  {sp+'/'+sub:<16}: exists={os.path.isdir(p)}")

In [ ]:
# ============================================================
# Cell 3 — Official EMCAD lib/pvtv2.py (PVTv2 encoder)
# ============================================================
# Copyright (c) 2021, The University of Texas at Austin. All rights reserved.
# Verbatim except the register_model line (marked [notebook edit]).

import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial

from timm.models.layers import DropPath, to_2tuple, trunc_normal_
# [notebook edit] timm already registers its own pvt_v2_* models; use a no-op
# decorator so the official code does not clash with timm's registry.
register_model = lambda fn: fn

import math


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.dwconv = DWConv(hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x, H, W):
        x = self.fc1(x)
        x = self.dwconv(x, H, W)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None, attn_drop=0., proj_drop=0., sr_ratio=1):
        super().__init__()
        assert dim % num_heads == 0, f"dim {dim} should be divided by num_heads {num_heads}."

        self.dim = dim
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        self.q = nn.Linear(dim, dim, bias=qkv_bias)
        self.kv = nn.Linear(dim, dim * 2, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        self.sr_ratio = sr_ratio
        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio)
            self.norm = nn.LayerNorm(dim)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x, H, W):
        B, N, C = x.shape
        q = self.q(x).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)

        if self.sr_ratio > 1:
            x_ = x.permute(0, 2, 1).reshape(B, C, H, W)
            x_ = self.sr(x_).reshape(B, C, -1).permute(0, 2, 1)
            x_ = self.norm(x_)
            kv = self.kv(x_).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        else:
            kv = self.kv(x).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)

        return x


class Block(nn.Module):

    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., act_layer=nn.GELU, norm_layer=nn.LayerNorm, sr_ratio=1):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(
            dim,
            num_heads=num_heads, qkv_bias=qkv_bias, qk_scale=qk_scale,
            attn_drop=attn_drop, proj_drop=drop, sr_ratio=sr_ratio)
        # NOTE: drop path for stochastic depth, we shall see if this is better than dropout here
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x, H, W):
        x = x + self.drop_path(self.attn(self.norm1(x), H, W))
        x = x + self.drop_path(self.mlp(self.norm2(x), H, W))

        return x


class OverlapPatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """

    def __init__(self, img_size=224, patch_size=7, stride=4, in_chans=3, embed_dim=768):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)

        self.img_size = img_size
        self.patch_size = patch_size
        self.H, self.W = img_size[0] // patch_size[0], img_size[1] // patch_size[1]
        self.num_patches = self.H * self.W
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=stride,
                              padding=(patch_size[0] // 2, patch_size[1] // 2))
        self.norm = nn.LayerNorm(embed_dim)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x):
        x = self.proj(x)
        _, _, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)

        return x, H, W


class PyramidVisionTransformerImpr(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000, embed_dims=[64, 128, 256, 512],
                 num_heads=[1, 2, 4, 8], mlp_ratios=[4, 4, 4, 4], qkv_bias=False, qk_scale=None, drop_rate=0.,
                 attn_drop_rate=0., drop_path_rate=0., norm_layer=nn.LayerNorm,
                 depths=[3, 4, 6, 3], sr_ratios=[8, 4, 2, 1]):
        super().__init__()
        self.num_classes = num_classes
        self.depths = depths

        # patch_embed
        self.patch_embed1 = OverlapPatchEmbed(img_size=img_size, patch_size=7, stride=4, in_chans=in_chans,
                                              embed_dim=embed_dims[0])
        self.patch_embed2 = OverlapPatchEmbed(img_size=img_size // 4, patch_size=3, stride=2, in_chans=embed_dims[0],
                                              embed_dim=embed_dims[1])
        self.patch_embed3 = OverlapPatchEmbed(img_size=img_size // 8, patch_size=3, stride=2, in_chans=embed_dims[1],
                                              embed_dim=embed_dims[2])
        self.patch_embed4 = OverlapPatchEmbed(img_size=img_size // 16, patch_size=3, stride=2, in_chans=embed_dims[2],
                                              embed_dim=embed_dims[3])

        # transformer encoder
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule
        cur = 0
        self.block1 = nn.ModuleList([Block(
            dim=embed_dims[0], num_heads=num_heads[0], mlp_ratio=mlp_ratios[0], qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[cur + i], norm_layer=norm_layer,
            sr_ratio=sr_ratios[0])
            for i in range(depths[0])])
        self.norm1 = norm_layer(embed_dims[0])

        cur += depths[0]
        self.block2 = nn.ModuleList([Block(
            dim=embed_dims[1], num_heads=num_heads[1], mlp_ratio=mlp_ratios[1], qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[cur + i], norm_layer=norm_layer,
            sr_ratio=sr_ratios[1])
            for i in range(depths[1])])
        self.norm2 = norm_layer(embed_dims[1])

        cur += depths[1]
        self.block3 = nn.ModuleList([Block(
            dim=embed_dims[2], num_heads=num_heads[2], mlp_ratio=mlp_ratios[2], qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[cur + i], norm_layer=norm_layer,
            sr_ratio=sr_ratios[2])
            for i in range(depths[2])])
        self.norm3 = norm_layer(embed_dims[2])

        cur += depths[2]
        self.block4 = nn.ModuleList([Block(
            dim=embed_dims[3], num_heads=num_heads[3], mlp_ratio=mlp_ratios[3], qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[cur + i], norm_layer=norm_layer,
            sr_ratio=sr_ratios[3])
            for i in range(depths[3])])
        self.norm4 = norm_layer(embed_dims[3])

        # classification head
        # self.head = nn.Linear(embed_dims[3], num_classes) if num_classes > 0 else nn.Identity()

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def init_weights(self, pretrained=None):
        if isinstance(pretrained, str):
            logger = 1
            #load_checkpoint(self, pretrained, map_location='cpu', strict=False, logger=logger)

    def reset_drop_path(self, drop_path_rate):
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(self.depths))]
        cur = 0
        for i in range(self.depths[0]):
            self.block1[i].drop_path.drop_prob = dpr[cur + i]

        cur += self.depths[0]
        for i in range(self.depths[1]):
            self.block2[i].drop_path.drop_prob = dpr[cur + i]

        cur += self.depths[1]
        for i in range(self.depths[2]):
            self.block3[i].drop_path.drop_prob = dpr[cur + i]

        cur += self.depths[2]
        for i in range(self.depths[3]):
            self.block4[i].drop_path.drop_prob = dpr[cur + i]

    def freeze_patch_emb(self):
        self.patch_embed1.requires_grad = False

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'pos_embed1', 'pos_embed2', 'pos_embed3', 'pos_embed4', 'cls_token'}  # has pos_embed may be better

    def get_classifier(self):
        return self.head

    def reset_classifier(self, num_classes, global_pool=''):
        self.num_classes = num_classes
        self.head = nn.Linear(self.embed_dim, num_classes) if num_classes > 0 else nn.Identity()

    # def _get_pos_embed(self, pos_embed, patch_embed, H, W):
    #     if H * W == self.patch_embed1.num_patches:
    #         return pos_embed
    #     else:
    #         return F.interpolate(
    #             pos_embed.reshape(1, patch_embed.H, patch_embed.W, -1).permute(0, 3, 1, 2),
    #             size=(H, W), mode="bilinear").reshape(1, -1, H * W).permute(0, 2, 1)

    def forward_features(self, x):
        B = x.shape[0]
        outs = []
        
        # stage 1
        x, H, W = self.patch_embed1(x)
        for i, blk in enumerate(self.block1):
            x = blk(x, H, W)
        x = self.norm1(x)
        x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        outs.append(x)

        # stage 2
        x, H, W = self.patch_embed2(x)
        for i, blk in enumerate(self.block2):
            x = blk(x, H, W)
        x = self.norm2(x)
        x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        outs.append(x)

        # stage 3
        x, H, W = self.patch_embed3(x)
        for i, blk in enumerate(self.block3):
            x = blk(x, H, W)
        x = self.norm3(x)
        x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        outs.append(x)

        # stage 4
        x, H, W = self.patch_embed4(x)
        for i, blk in enumerate(self.block4):
            x = blk(x, H, W)
        x = self.norm4(x)
        x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        outs.append(x)

        return outs

        # return x.mean(dim=1)

    def forward(self, x):
        x = self.forward_features(x)
        # x = self.head(x)

        return x


class DWConv(nn.Module):
    def __init__(self, dim=768):
        super(DWConv, self).__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, bias=True, groups=dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1, 2).view(B, C, H, W)
        x = self.dwconv(x)
        x = x.flatten(2).transpose(1, 2)

        return x


def _conv_filter(state_dict, patch_size=16):
    """ convert patch embedding weight from manual patchify + linear proj to conv"""
    out_dict = {}
    for k, v in state_dict.items():
        if 'patch_embed.proj.weight' in k:
            v = v.reshape((v.shape[0], 3, patch_size, patch_size))
        out_dict[k] = v

    return out_dict


@register_model
class pvt_v2_b0(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b0, self).__init__(
            patch_size=4, embed_dims=[32, 64, 160, 256], num_heads=[1, 2, 5, 8], mlp_ratios=[8, 8, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[2, 2, 2, 2], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)



@register_model
class pvt_v2_b1(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b1, self).__init__(
            patch_size=4, embed_dims=[64, 128, 320, 512], num_heads=[1, 2, 5, 8], mlp_ratios=[8, 8, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[2, 2, 2, 2], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)

@register_model
class pvt_v2_b2(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b2, self).__init__(
            patch_size=4, embed_dims=[64, 128, 320, 512], num_heads=[1, 2, 5, 8], mlp_ratios=[8, 8, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 4, 6, 3], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)

@register_model
class pvt_v2_b3(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b3, self).__init__(
            patch_size=4, embed_dims=[64, 128, 320, 512], num_heads=[1, 2, 5, 8], mlp_ratios=[8, 8, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 4, 18, 3], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)

@register_model
class pvt_v2_b4(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b4, self).__init__(
            patch_size=4, embed_dims=[64, 128, 320, 512], num_heads=[1, 2, 5, 8], mlp_ratios=[8, 8, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 8, 27, 3], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)


@register_model
class pvt_v2_b5(PyramidVisionTransformerImpr):
    def __init__(self, **kwargs):
        super(pvt_v2_b5, self).__init__(
            patch_size=4, embed_dims=[64, 128, 320, 512], num_heads=[1, 2, 5, 8], mlp_ratios=[4, 4, 4, 4],
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[3, 6, 40, 3], sr_ratios=[8, 4, 2, 1],
            drop_rate=0.0, drop_path_rate=0.1)

print('PVTv2 defined.')

In [ ]:
# ============================================================
# Cell 4 — Official EMCAD lib/decoders.py (MSCB, EUCB, LGAG, CAB, SAB, EMCAD)
# ============================================================
# Copyright (c) 2021, The University of Texas at Austin. All rights reserved.
# Verbatim.

import torch
import torch.nn as nn
from functools import partial

import math
from timm.models.layers import trunc_normal_tf_
from timm.models.helpers import named_apply


def gcd(a, b):
    while b:
        a, b = b, a % b
    return a

# Other types of layers can go here (e.g., nn.Linear, etc.)
def _init_weights(module, name, scheme=''):
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Conv3d):
        if scheme == 'normal':
            nn.init.normal_(module.weight, std=.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif scheme == 'trunc_normal':
            trunc_normal_tf_(module.weight, std=.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif scheme == 'xavier_normal':
            nn.init.xavier_normal_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif scheme == 'kaiming_normal':
            nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        else:
            # efficientnet like
            fan_out = module.kernel_size[0] * module.kernel_size[1] * module.out_channels
            fan_out //= module.groups
            nn.init.normal_(module.weight, 0, math.sqrt(2.0 / fan_out))
            if module.bias is not None:
                nn.init.zeros_(module.bias)
    elif isinstance(module, nn.BatchNorm2d) or isinstance(module, nn.BatchNorm3d):
        nn.init.constant_(module.weight, 1)
        nn.init.constant_(module.bias, 0)
    elif isinstance(module, nn.LayerNorm):
        nn.init.constant_(module.weight, 1)
        nn.init.constant_(module.bias, 0)

def act_layer(act, inplace=False, neg_slope=0.2, n_prelu=1):
    # activation layer
    act = act.lower()
    if act == 'relu':
        layer = nn.ReLU(inplace)
    elif act == 'relu6':
        layer = nn.ReLU6(inplace)
    elif act == 'leakyrelu':
        layer = nn.LeakyReLU(neg_slope, inplace)
    elif act == 'prelu':
        layer = nn.PReLU(num_parameters=n_prelu, init=neg_slope)
    elif act == 'gelu':
        layer = nn.GELU()
    elif act == 'hswish':
        layer = nn.Hardswish(inplace)
    else:
        raise NotImplementedError('activation layer [%s] is not found' % act)
    return layer

def channel_shuffle(x, groups):
    batchsize, num_channels, height, width = x.data.size()
    channels_per_group = num_channels // groups    
    # reshape
    x = x.view(batchsize, groups, 
               channels_per_group, height, width)
    x = torch.transpose(x, 1, 2).contiguous()
    # flatten
    x = x.view(batchsize, -1, height, width)
    return x

#   Multi-scale depth-wise convolution (MSDC)
class MSDC(nn.Module):
    def __init__(self, in_channels, kernel_sizes, stride, activation='relu6', dw_parallel=True):
        super(MSDC, self).__init__()

        self.in_channels = in_channels
        self.kernel_sizes = kernel_sizes
        self.activation = activation
        self.dw_parallel = dw_parallel

        self.dwconvs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(self.in_channels, self.in_channels, kernel_size, stride, kernel_size // 2, groups=self.in_channels, bias=False),
                nn.BatchNorm2d(self.in_channels),
                act_layer(self.activation, inplace=True)
            )
            for kernel_size in self.kernel_sizes
        ])

        self.init_weights('normal')
    
    def init_weights(self, scheme=''):
        named_apply(partial(_init_weights, scheme=scheme), self)

    def forward(self, x):
        # Apply the convolution layers in a loop
        outputs = []
        for dwconv in self.dwconvs:
            dw_out = dwconv(x)
            outputs.append(dw_out)
            if self.dw_parallel == False:
                x = x+dw_out
        # You can return outputs based on what you intend to do with them
        return outputs

class MSCB(nn.Module):
    """
    Multi-scale convolution block (MSCB) 
    """
    def __init__(self, in_channels, out_channels, stride, kernel_sizes=[1,3,5], expansion_factor=2, dw_parallel=True, add=True, activation='relu6'):
        super(MSCB, self).__init__()
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride
        self.kernel_sizes = kernel_sizes
        self.expansion_factor = expansion_factor
        self.dw_parallel = dw_parallel
        self.add = add
        self.activation = activation
        self.n_scales = len(self.kernel_sizes)
        # check stride value
        assert self.stride in [1, 2]
        # Skip connection if stride is 1
        self.use_skip_connection = True if self.stride == 1 else False

        # expansion factor
        self.ex_channels = int(self.in_channels * self.expansion_factor)
        self.pconv1 = nn.Sequential(
            # pointwise convolution
            nn.Conv2d(self.in_channels, self.ex_channels, 1, 1, 0, bias=False),
            nn.BatchNorm2d(self.ex_channels),
            act_layer(self.activation, inplace=True)
        )
        self.msdc = MSDC(self.ex_channels, self.kernel_sizes, self.stride, self.activation, dw_parallel=self.dw_parallel)
        if self.add == True:
            self.combined_channels = self.ex_channels*1
        else:
            self.combined_channels = self.ex_channels*self.n_scales
        self.pconv2 = nn.Sequential(
            # pointwise convolution
            nn.Conv2d(self.combined_channels, self.out_channels, 1, 1, 0, bias=False), 
            nn.BatchNorm2d(self.out_channels),
        )
        if self.use_skip_connection and (self.in_channels != self.out_channels):
            self.conv1x1 = nn.Conv2d(self.in_channels, self.out_channels, 1, 1, 0, bias=False)
        self.init_weights('normal')
    
    def init_weights(self, scheme=''):
        named_apply(partial(_init_weights, scheme=scheme), self)

    def forward(self, x):
        pout1 = self.pconv1(x)
        msdc_outs = self.msdc(pout1)
        if self.add == True:
            dout = 0
            for dwout in msdc_outs:
                dout = dout + dwout
        else:
            dout = torch.cat(msdc_outs, dim=1)
        dout = channel_shuffle(dout, gcd(self.combined_channels,self.out_channels))
        out = self.pconv2(dout)
        if self.use_skip_connection:
            if self.in_channels != self.out_channels:
                x = self.conv1x1(x)
            return x + out
        else:
            return out
        
#   Multi-scale convolution block (MSCB)
def MSCBLayer(in_channels, out_channels, n=1, stride=1, kernel_sizes=[1,3,5], expansion_factor=2, dw_parallel=True, add=True, activation='relu6'):
        """
        create a series of multi-scale convolution blocks.
        """
        convs = []
        mscb = MSCB(in_channels, out_channels, stride, kernel_sizes=kernel_sizes, expansion_factor=expansion_factor, dw_parallel=dw_parallel, add=add, activation=activation)
        convs.append(mscb)
        if n > 1:
            for i in range(1, n):
                mscb = MSCB(out_channels, out_channels, 1, kernel_sizes=kernel_sizes, expansion_factor=expansion_factor, dw_parallel=dw_parallel, add=add, activation=activation)
                convs.append(mscb)
        conv = nn.Sequential(*convs)
        return conv

#   Efficient up-convolution block (EUCB)
class EUCB(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, activation='relu'):
        super(EUCB,self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.up_dwc = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(self.in_channels, self.in_channels, kernel_size=kernel_size, stride=stride, padding=kernel_size//2, groups=self.in_channels, bias=False),
                nn.BatchNorm2d(self.in_channels),
            act_layer(activation, inplace=True)
        )
        self.pwc = nn.Sequential(
            nn.Conv2d(self.in_channels, self.out_channels, kernel_size=1, stride=1, padding=0, bias=True)
        ) 
        self.init_weights('normal')
    
    def init_weights(self, scheme=''):
        named_apply(partial(_init_weights, scheme=scheme), self)

    def forward(self, x):
        x = self.up_dwc(x)
        x = channel_shuffle(x, self.in_channels)
        x = self.pwc(x)
        return x

#   Large-kernel grouped attention gate (LGAG)
class LGAG(nn.Module):
    def __init__(self, F_g, F_l, F_int, kernel_size=3, groups=1, activation='relu'):
        super(LGAG,self).__init__()

        if kernel_size == 1:
            groups = 1
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=kernel_size, stride=1, padding=kernel_size//2, groups=groups, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=kernel_size, stride=1, padding=kernel_size//2, groups=groups, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1,stride=1,padding=0,bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.activation = act_layer(activation, inplace=True)

        self.init_weights('normal')
    
    def init_weights(self, scheme=''):
        named_apply(partial(_init_weights, scheme=scheme), self)
                
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.activation(g1 + x1)
        psi = self.psi(psi)

        return x*psi
    
#   Channel attention block (CAB)
class CAB(nn.Module):
    def __init__(self, in_channels, out_channels=None, ratio=16, activation='relu'):
        super(CAB, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        if self.in_channels < ratio:
            ratio = self.in_channels
        self.reduced_channels = self.in_channels // ratio
        if self.out_channels == None:
            self.out_channels = in_channels

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.activation = act_layer(activation, inplace=True)
        self.fc1 = nn.Conv2d(self.in_channels, self.reduced_channels, 1, bias=False)
        self.fc2 = nn.Conv2d(self.reduced_channels, self.out_channels, 1, bias=False)
        
        self.sigmoid = nn.Sigmoid()

        self.init_weights('normal')
    
    def init_weights(self, scheme=''):
        named_apply(partial(_init_weights, scheme=scheme), self)

    def forward(self, x):
        avg_pool_out = self.avg_pool(x) 
        avg_out = self.fc2(self.activation(self.fc1(avg_pool_out)))

        max_pool_out= self.max_pool(x) 
        max_out = self.fc2(self.activation(self.fc1(max_pool_out)))

        out = avg_out + max_out
        return self.sigmoid(out) 
    
#   Spatial attention block (SAB)
class SAB(nn.Module):
    def __init__(self, kernel_size=7):
        super(SAB, self).__init__()

        assert kernel_size in (3, 7, 11), 'kernel must be 3 or 7 or 11'
        padding = kernel_size//2

        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
           
        self.sigmoid = nn.Sigmoid()

        self.init_weights('normal')
    
    def init_weights(self, scheme=''):
        named_apply(partial(_init_weights, scheme=scheme), self)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv(x)
        return self.sigmoid(x)

#   Efficient multi-scale convolutional attention decoding (EMCAD)
class EMCAD(nn.Module):
    def __init__(self, channels=[512,320,128,64], kernel_sizes=[1,3,5], expansion_factor=6, dw_parallel=True, add=True, lgag_ks=3, activation='relu6'):
        super(EMCAD,self).__init__()
        eucb_ks = 3 # kernel size for eucb
        self.mscb4 = MSCBLayer(channels[0], channels[0], n=1, stride=1, kernel_sizes=kernel_sizes, expansion_factor=expansion_factor, dw_parallel=dw_parallel, add=add, activation=activation)
        
        self.eucb3 = EUCB(in_channels=channels[0], out_channels=channels[1], kernel_size=eucb_ks, stride=eucb_ks//2)
        self.lgag3 = LGAG(F_g=channels[1], F_l=channels[1], F_int=channels[1]//2, kernel_size=lgag_ks, groups=channels[1]//2)
        self.mscb3 = MSCBLayer(channels[1], channels[1], n=1, stride=1, kernel_sizes=kernel_sizes, expansion_factor=expansion_factor, dw_parallel=dw_parallel, add=add, activation=activation)

        self.eucb2 = EUCB(in_channels=channels[1], out_channels=channels[2], kernel_size=eucb_ks, stride=eucb_ks//2)
        self.lgag2 = LGAG(F_g=channels[2], F_l=channels[2], F_int=channels[2]//2, kernel_size=lgag_ks, groups=channels[2]//2)
        self.mscb2 = MSCBLayer(channels[2], channels[2], n=1, stride=1, kernel_sizes=kernel_sizes, expansion_factor=expansion_factor, dw_parallel=dw_parallel, add=add, activation=activation)
        
        self.eucb1 = EUCB(in_channels=channels[2], out_channels=channels[3], kernel_size=eucb_ks, stride=eucb_ks//2)
        self.lgag1 = LGAG(F_g=channels[3], F_l=channels[3], F_int=int(channels[3]/2), kernel_size=lgag_ks, groups=int(channels[3]/2))
        self.mscb1 = MSCBLayer(channels[3], channels[3], n=1, stride=1, kernel_sizes=kernel_sizes, expansion_factor=expansion_factor, dw_parallel=dw_parallel, add=add, activation=activation)
        
        self.cab4 = CAB(channels[0])
        self.cab3 = CAB(channels[1])
        self.cab2 = CAB(channels[2])
        self.cab1 = CAB(channels[3])
        
        self.sab = SAB()
       
      
    def forward(self, x, skips):
            
        # MSCAM4
        d4 = self.cab4(x)*x
        d4 = self.sab(d4)*d4 
        d4 = self.mscb4(d4)
        
        # EUCB3
        d3 = self.eucb3(d4)
                
        # LGAG3
        x3 = self.lgag3(g=d3, x=skips[0])
        
        # Additive aggregation 3
        d3 = d3 + x3
        
        # MSCAM3
        d3 = self.cab3(d3)*d3
        d3 = self.sab(d3)*d3  
        d3 = self.mscb3(d3)
        
        # EUCB2
        d2 = self.eucb2(d3)
        
        # LGAG2
        x2 = self.lgag2(g=d2, x=skips[1])
        
        # Additive aggregation 2
        d2 = d2 + x2 
        
        # MSCAM2
        d2 = self.cab2(d2)*d2
        d2 = self.sab(d2)*d2
        d2 = self.mscb2(d2)
        
        # EUCB1
        d1 = self.eucb1(d2)
        
        # LGAG1
        x1 = self.lgag1(g=d1, x=skips[2])
        
        # Additive aggregation 1
        d1 = d1 + x1 
        
        # MSCAM1
        d1 = self.cab1(d1)*d1
        d1 = self.sab(d1)*d1
        d1 = self.mscb1(d1)
        
        return [d4, d3, d2, d1]

print('EMCAD decoder defined.')

In [ ]:
# ============================================================
# Cell 5 — Official EMCAD lib/networks.py (EMCADNet)
# ============================================================
# Copyright (c) 2021, The University of Texas at Austin. All rights reserved.
# Verbatim except: ResNet branches removed (not used), torch.load map_location.
# forward() returns [p4, p3, p2, p1]; all upsampled to input size.

import torch
import torch.nn as nn
import torch.nn.functional as F



class EMCADNet(nn.Module):
    def __init__(self, num_classes=1, kernel_sizes=[1,3,5], expansion_factor=2, dw_parallel=True, add=True, lgag_ks=3, activation='relu', encoder='pvt_v2_b2', pretrain=True, pretrained_dir='./pretrained_pth/pvt/'):
        super(EMCADNet, self).__init__()

        # conv block to convert single channel to 3 channels
        self.conv = nn.Sequential(
            nn.Conv2d(1, 3, kernel_size=1),
            nn.BatchNorm2d(3),
            nn.ReLU(inplace=True)
        )
        
        # backbone network initialization with pretrained weight
        if encoder == 'pvt_v2_b0':
            self.backbone = pvt_v2_b0()
            path = pretrained_dir + '/pvt_v2_b0.pth'
            channels=[256, 160, 64, 32]
        elif encoder == 'pvt_v2_b1':
            self.backbone = pvt_v2_b1()
            path = pretrained_dir + '/pvt_v2_b1.pth'
            channels=[512, 320, 128, 64]
        elif encoder == 'pvt_v2_b2':
            self.backbone = pvt_v2_b2()
            path = pretrained_dir + '/pvt_v2_b2.pth'
            channels=[512, 320, 128, 64]
        elif encoder == 'pvt_v2_b3':
            self.backbone = pvt_v2_b3()
            path = pretrained_dir + '/pvt_v2_b3.pth'
            channels=[512, 320, 128, 64]
        elif encoder == 'pvt_v2_b4':
            self.backbone = pvt_v2_b4()
            path = pretrained_dir + '/pvt_v2_b4.pth'
            channels=[512, 320, 128, 64]
        elif encoder == 'pvt_v2_b5':
            self.backbone = pvt_v2_b5() 
            path = pretrained_dir + '/pvt_v2_b5.pth'
            channels=[512, 320, 128, 64]
        else:
            print('Encoder not implemented! Continuing with default encoder pvt_v2_b2.')
            self.backbone = pvt_v2_b2()  
            path = pretrained_dir + '/pvt_v2_b2.pth'
            channels=[512, 320, 128, 64]
            
        if pretrain==True and 'pvt_v2' in encoder:
            save_model = torch.load(path, map_location='cpu')  # [notebook edit] map_location
            model_dict = self.backbone.state_dict()
            state_dict = {k: v for k, v in save_model.items() if k in model_dict.keys()}
            model_dict.update(state_dict)
            self.backbone.load_state_dict(model_dict)
        
        print('Model %s created, param count: %d' %
                     (encoder+' backbone: ', sum([m.numel() for m in self.backbone.parameters()])))
        
        #   decoder initialization
        self.decoder = EMCAD(channels=channels, kernel_sizes=kernel_sizes, expansion_factor=expansion_factor, dw_parallel=dw_parallel, add=add, lgag_ks=lgag_ks, activation=activation)
        
        print('Model %s created, param count: %d' %
                     ('EMCAD decoder: ', sum([m.numel() for m in self.decoder.parameters()])))
             
        self.out_head4 = nn.Conv2d(channels[0], num_classes, 1)
        self.out_head3 = nn.Conv2d(channels[1], num_classes, 1)
        self.out_head2 = nn.Conv2d(channels[2], num_classes, 1)
        self.out_head1 = nn.Conv2d(channels[3], num_classes, 1)
        
    def forward(self, x, mode='test'):
        
        # if grayscale input, convert to 3 channels
        if x.size()[1] == 1:
            x = self.conv(x)
        
        # encoder
        x1, x2, x3, x4 = self.backbone(x)
        #print(x1.shape, x2.shape, x3.shape, x4.shape)

        # decoder
        dec_outs = self.decoder(x4, [x3, x2, x1])
        
        # prediction heads  
        p4 = self.out_head4(dec_outs[0])
        p3 = self.out_head3(dec_outs[1])
        p2 = self.out_head2(dec_outs[2])
        p1 = self.out_head1(dec_outs[3])

        p4 = F.interpolate(p4, scale_factor=32, mode='bilinear')
        p3 = F.interpolate(p3, scale_factor=16, mode='bilinear')
        p2 = F.interpolate(p2, scale_factor=8, mode='bilinear')
        p1 = F.interpolate(p1, scale_factor=4, mode='bilinear')

        if mode == 'test':
            return [p4, p3, p2, p1]
        
        return [p4, p3, p2, p1]

print('EMCADNet defined.')

In [ ]:
# ============================================================
# Cell 6 — Official EMCAD utils/dataloader_polyp.py
# ============================================================
# Copyright (c) 2021, The University of Texas at Austin. All rights reserved.
# Verbatim. Train returns (image, mask); val/test return (image, mask, (W,H), name).

import os
import cv2
import numpy as np
import torch
import torch.utils.data as data
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

class PolypDataset(data.Dataset):
    """
    Unified adaptive dataloader for polyp segmentation.
    Uses Albumentations and strictly handles binary mask conversion.
    """
    def __init__(self, image_root, gt_root, trainsize, augmentation, split='train', color_image=True):
        self.trainsize = trainsize
        self.color_image = color_image
        self.augmentation = augmentation
        self.split = split
        
        # Load and sort file paths
        exts = ('.jpg', '.png', '.jpeg', '.tif')
        self.images = sorted([os.path.join(image_root, f) for f in os.listdir(image_root) if f.lower().endswith(exts)])
        self.gts = sorted([os.path.join(gt_root, f) for f in os.listdir(gt_root) if f.lower().endswith(exts)])
        
        self.filter_files()
        self.size = len(self.images)

        # Transformation Setup
        mean = [0.485, 0.456, 0.406] if color_image else [0.5]
        std = [0.229, 0.224, 0.225] if color_image else [0.229]

        if self.split == 'train' and self.augmentation:
            self.transform = A.Compose([
                A.Rotate(limit=90, p=0.5),
                A.VerticalFlip(p=0.5),
                A.HorizontalFlip(p=0.5),
                A.Resize(height=self.trainsize, width=self.trainsize),
                A.Normalize(mean=mean, std=std),
                ToTensorV2()
            ])
        else:
            self.transform = A.Compose([
                A.Resize(height=self.trainsize, width=self.trainsize),
                A.Normalize(mean=mean, std=std),
                ToTensorV2()
            ])

    def filter_files(self):
        valid_images, valid_gts = [], []
        for img_p, gt_p in zip(self.images, self.gts):
            if os.path.exists(img_p) and os.path.exists(gt_p):
                valid_images.append(img_p)
                valid_gts.append(gt_p)
        self.images, self.gts = valid_images, valid_gts

    def __getitem__(self, index):
        # 1. Load Image
        image = cv2.imread(self.images[index])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB if self.color_image else cv2.COLOR_BGR2GRAY)
        
        # 2. Load Mask
        mask_np = cv2.imread(self.gts[index], cv2.IMREAD_GRAYSCALE)

        # 3. Apply Transformations
        augmented = self.transform(image=image, mask=mask_np)
        image = augmented['image']
        mask = augmented['mask']

        # 4. Adaptive Binary Mask Logic
        # Thinking carefully: This handles 0/255 with noise and 0/1/2/3 labels.
        max_val = mask.max()
        if max_val > 127.0:
            # Treats everything above 20 as foreground to catch 255 but ignore noise
            mask = (mask > 20).long()
        else:
            # Treats all integer labels (1, 2, 3...) as foreground
            mask = (mask >= 1).long()

        if len(mask.shape) == 2:
            mask = mask.unsqueeze(0)

        # 5. Return Logic
        if self.split == 'train':
            return image, mask
        else:
            with Image.open(self.gts[index]) as img:
                original_shape = img.size
            name = os.path.basename(self.images[index])
            if name.lower().endswith('.jpg'):
                name = name.rsplit('.', 1)[0] + '.png'
            return image, mask, original_shape, name

    def __len__(self):
        return self.size

def get_loader(image_root, gt_root, batchsize, trainsize, shuffle=False, num_workers=4, pin_memory=True, augmentation=False, split='train', color_image=True):
    dataset = PolypDataset(image_root, gt_root, trainsize, augmentation, split, color_image)
    return data.DataLoader(dataset=dataset, batch_size=batchsize, shuffle=shuffle, num_workers=num_workers, pin_memory=pin_memory)
    

def check_split(root):
    """Images and masks are paired by sorted order in the official loader;
    make sure the stems really match, and print the split sizes."""
    exts = ('.jpg', '.png', '.jpeg', '.tif')
    sizes = {}
    for sp in ("train", "val", "test"):
        im = sorted(f for f in os.listdir(os.path.join(root, sp, "images")) if f.lower().endswith(exts))
        gt = sorted(f for f in os.listdir(os.path.join(root, sp, "masks"))  if f.lower().endswith(exts))
        assert len(im) == len(gt), f"{sp}: {len(im)} images vs {len(gt)} masks"
        bad = [(a, b) for a, b in zip(im, gt) if os.path.splitext(a)[0] != os.path.splitext(b)[0]]
        assert not bad, f"{sp}: image/mask names differ, e.g. {bad[:3]}"
        sizes[sp] = len(im)
    tr = set(os.listdir(os.path.join(root, "train", "images")))
    te = set(os.listdir(os.path.join(root, "test", "images")))
    print(f"EMCAD split: train {sizes['train']} | val {sizes['val']} | test {sizes['test']}  "
          f"(total {sum(sizes.values())}, train/test name overlap {len(tr & te)})")
    return sizes

In [ ]:
# ============================================================
# Cell 7 — Loss (official structure_loss) + metrics + official eval
# ============================================================
def structure_loss(pred, mask, w=1):
    """Verbatim from train_polyp.py: weighted BCE + weighted IoU, logits in."""
    weit = 1 + 5 * torch.abs(F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask)
    wbce = F.binary_cross_entropy_with_logits(pred, mask, reduction='none')
    wbce = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3))
    pred = torch.sigmoid(pred)
    inter = ((pred * mask) * weit).sum(dim=(2, 3))
    union = ((pred + mask) * weit).sum(dim=(2, 3))
    wiou = 1 - (inter + 1) / (union - inter + 1)
    return (w * (wbce + wiou)).mean()

def emcad_total_loss(P, gts):
    """train_polyp.py: loss on p4, p3, p2, p1 and on p4+p3+p2+p1, weights all 1."""
    P = [p.float() for p in P]
    return (structure_loss(P[0], gts) + structure_loss(P[1], gts) + structure_loss(P[2], gts)
            + structure_loss(P[3], gts) + structure_loss(P[0] + P[1] + P[2] + P[3], gts))

def per_image_metrics(pb, tb):
    """pb, tb: float {0,1} tensors. Dice/IoU equal the official functions (smooth 1e-6)."""
    TP = (pb * tb).sum().item(); FP = (pb * (1 - tb)).sum().item()
    FN = ((1 - pb) * tb).sum().item(); TN = ((1 - pb) * (1 - tb)).sum().item()
    s = 1e-6
    return {"dice": (2*TP + s) / (2*TP + FP + FN + s), "iou": (TP + s) / (TP + FP + FN + s),
            "sensitivity": (TP + s) / (TP + FN + s), "specificity": (TN + s) / (TN + FP + s),
            "precision": (TP + s) / (TP + FP + s), "accuracy": (TP + TN + s) / (TP + TN + FP + FN + s)}

METRIC_KEYS = ["dice", "iou", "sensitivity", "specificity", "precision", "accuracy"]

def make_eval_loader(split):
    root = os.path.join(CONFIG["clinicdb_root"], split)
    return get_loader(image_root=f"{root}/images/", gt_root=f"{root}/masks/",
                      batchsize=CONFIG["test_batch_size"], trainsize=CONFIG["img_size"],
                      shuffle=False, num_workers=CONFIG["num_workers"], split="test",
                      color_image=CONFIG["color_image"])

@torch.no_grad()
def evaluate(model, loader, orientation="official", desc="Eval"):
    """Official test(): last head p1, bilinear resize, sigmoid, per-image min-max,
    pred >= 0.5, GT (352 px, nearest-resized) >= 0.2.
    orientation="official"  -> size=(W, H) exactly as train_polyp.py does
    orientation="corrected" -> size=(H, W)"""
    model.eval()
    rows = []
    for images, gts, shapes, names in tqdm(loader, desc=desc, leave=False):
        images = images.to(DEVICE); gts = gts.to(DEVICE).float()
        pred = model(images)
        pred = pred[-1] if isinstance(pred, list) else pred
        for i in range(images.size(0)):
            W, H = int(shapes[0][i]), int(shapes[1][i])
            size = (W, H) if orientation == "official" else (H, W)
            p = F.interpolate(pred[i:i+1].float(), size=size, mode="bilinear", align_corners=False).sigmoid().squeeze()
            p = (p - p.min()) / (p.max() - p.min() + 1e-8)
            g = F.interpolate(gts[i:i+1], size=size, mode="nearest").squeeze()
            m = per_image_metrics((p >= 0.5).float(), (g >= 0.2).float())
            m["image"] = names[i]
            rows.append(m)
    df = pd.DataFrame(rows)
    return {k: float(df[k].mean()) for k in METRIC_KEYS}, df

print("structure_loss, metrics, official evaluate() defined.")

In [ ]:
# ============================================================
# Cell 8 — Build model, sanity checks, smoke test + time estimate
# ============================================================
def build_model(cfg):
    return EMCADNet(num_classes=1, kernel_sizes=cfg["kernel_sizes"],
                    expansion_factor=cfg["expansion_factor"], dw_parallel=cfg["dw_parallel"],
                    add=cfg["add"], lgag_ks=cfg["lgag_ks"], activation=cfg["activation_mscb"],
                    encoder=cfg["encoder"], pretrain=True, pretrained_dir=cfg["pretrained_dir"])

sizes = check_split(CONFIG["clinicdb_root"])

_m = build_model(CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in _m.parameters())
n_lgag = sum(p.numel() for n, p in _m.named_parameters() if ".lgag" in n)
print(f"Params total : {n_params:,}  (EMCAD paper B0: 3.92M; official code: 3,916,795)")
print(f"  LGAG gates : {n_lgag:,}  (lgag3 @1/16, lgag2 @1/8, lgag1 @1/4)")

_m.eval()
with torch.no_grad():
    _P = _m(torch.randn(1, 3, CONFIG["img_size"], CONFIG["img_size"], device=DEVICE))
print("Outputs      :", [tuple(p.shape) for p in _P], "(p4, p3, p2, p1)")
try:
    import copy
    _mc = copy.deepcopy(_m).cpu()
    _fl, _ = thop_profile(_mc, inputs=(torch.randn(1, 3, CONFIG["img_size"], CONFIG["img_size"]),), verbose=False)
    print(f"FLOPs (thop) : {_fl/1e9:.3f} G MACs @ {CONFIG['img_size']}  (same counter as EMCAD's cal_params_flops)")
    del _mc
except Exception as e:
    print(f"thop failed: {e}")

# Smoke test: forward + backward at every training scale, timed.
_m.train()
_opt = torch.optim.AdamW(_m.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
bs, S0 = CONFIG["batch_size"], CONFIG["img_size"]
sync = (lambda: torch.cuda.synchronize()) if DEVICE.type == "cuda" else (lambda: None)
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
t_scale = {}
for rate in CONFIG["size_rates"]:
    S = S0 if rate == 1 else int(round(S0 * rate / 32) * 32)
    x = torch.randn(bs, 3, S, S, device=DEVICE)
    y = (torch.rand(bs, 1, S, S, device=DEVICE) > 0.5).float()
    times = []
    for it in range(3):
        sync(); t0 = time.time()
        _opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, enabled=CONFIG["use_amp"] and DEVICE.type == "cuda"):
            P = _m(x)
        loss = emcad_total_loss(P, y)
        loss.backward(); _opt.step(); sync()
        if it > 0:
            times.append(time.time() - t0)
    t_scale[S] = float(np.mean(times))
    print(f"  scale {rate:<4} ({S}x{S}) : {t_scale[S]*1000:6.0f} ms/step  loss {loss.item():.3f}  OK")
step = sum(t_scale.values())
n_batches = math.ceil(sizes["train"] / bs)
est = step * n_batches + 10
print(f"Peak memory  : {torch.cuda.max_memory_allocated()/1e9:.2f} GB" if DEVICE.type == "cuda" else "")
print(f"Estimate     : ~{est:.0f} s/epoch -> {CONFIG['epochs']} epochs ~ {est*CONFIG['epochs']/3600:.1f} h "
      f"(random tensors, 'fixed'-mode sizes; official mode trains 256/256/448)")
del _m, _opt, _P
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Cell 9 — Loaders (official get_loader)
# ============================================================
seed_everything(SEED)
_tr = os.path.join(CONFIG["clinicdb_root"], "train")
train_loader = get_loader(image_root=f"{_tr}/images/", gt_root=f"{_tr}/masks/",
                          batchsize=CONFIG["batch_size"], trainsize=CONFIG["img_size"], shuffle=True,
                          num_workers=CONFIG["num_workers"], augmentation=CONFIG["augmentation"],
                          split="train", color_image=CONFIG["color_image"])
val_loader  = make_eval_loader("val")
test_loader = make_eval_loader("test")
print(f"train {len(train_loader.dataset)} ({len(train_loader)} batches) | "
      f"val {len(val_loader.dataset)} | test {len(test_loader.dataset)}")

In [ ]:
# ============================================================
# Cell 10 — Model, optimizer, scheduler, resume
# ============================================================
seed_everything(SEED)
model = build_model(CONFIG).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"],
                                                       eta_min=CONFIG["cosine_eta_min"])
scaler = torch.amp.GradScaler("cuda", enabled=CONFIG["use_amp"] and DEVICE.type == "cuda")
# Note: train_polyp.py also calls adjust_lr(decay_epoch=300); with 200 epochs it
# multiplies the LR by 0.1**0 = 1 every epoch, i.e. a no-op, so it is omitted.

ckpt_dir, out_dir = CONFIG["ckpt_dir"], CONFIG["output_dir"]
csv_path = os.path.join(out_dir, "training_log.csv")
CSV_HEADER = ["epoch", "train_loss"] + [f"val_{k}" for k in METRIC_KEYS] + ["lr", "epoch_time_sec", "gpu_mem_peak_gb"]

start_epoch, best_val, best_epoch, history, total_sec = 1, 0.0, 0, [], 0.0
if RESUME and RESUME_CKPT and os.path.exists(RESUME_CKPT):
    ck = torch.load(RESUME_CKPT, map_location=DEVICE, weights_only=False)
    assert ck["config"]["slug"] == SLUG, f"checkpoint is from {ck['config']['slug']}, not {SLUG}"
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    start_epoch, best_val, best_epoch = ck["epoch"] + 1, ck["best_val"], ck["best_epoch"]
    history, total_sec = ck["history"], ck["total_sec"]
    _best_src = os.path.join(os.path.dirname(RESUME_CKPT), "best.pth")   # best.pth from the same version
    if os.path.exists(_best_src) and not os.path.exists(os.path.join(ckpt_dir, "best.pth")):
        import shutil; shutil.copy(_best_src, os.path.join(ckpt_dir, "best.pth"))
    print(f"Resumed at epoch {start_epoch}, best val {best_val:.4f} @ {best_epoch}")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f); w.writerow(CSV_HEADER)
    for r in history:
        w.writerow([r[k] for k in CSV_HEADER])
print(f"Params {sum(p.numel() for p in model.parameters()):,} | AdamW lr {CONFIG['lr']} wd {CONFIG['weight_decay']} | "
      f"cosine T_max={CONFIG['epochs']} eta_min={CONFIG['cosine_eta_min']} | AMP {scaler.is_enabled()}")

In [ ]:
# ============================================================
# Cell 11 — Training (val every epoch, test ONCE after training)
# ============================================================
# Save a Kaggle version when done; /kaggle/working is lost on timeout.

def train_one_epoch():
    model.train()
    losses = []
    for images, gts in tqdm(train_loader, desc="  train", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        gts = gts.to(DEVICE, non_blocking=True).float()
        cur_img, cur_gt = images, gts
        for rate in CONFIG["size_rates"]:
            # official: the resized batch from the previous rate is reused (see top cell)
            base_img, base_gt = (cur_img, cur_gt) if CONFIG["multiscale_mode"] == "official" else (images, gts)
            if rate != 1:
                ts = int(round(CONFIG["img_size"] * rate / 32) * 32)
                x = F.interpolate(base_img, size=(ts, ts), mode="bilinear", align_corners=True)
                y = F.interpolate(base_gt, size=(ts, ts), mode="nearest")
            else:
                x, y = base_img, base_gt
            cur_img, cur_gt = x, y
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=scaler.is_enabled()):
                P = model(x)
            loss = emcad_total_loss(P, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            for g in optimizer.param_groups:                      # official clip_gradient: value clamp
                for p in g["params"]:
                    if p.grad is not None:
                        p.grad.data.clamp_(-CONFIG["clip"], CONFIG["clip"])
            scaler.step(optimizer); scaler.update()
            if rate == 1:
                losses.append(loss.item())
    return float(np.mean(losses))

def save_ckpt(path, epoch):
    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "best_val": best_val, "best_epoch": best_epoch, "history": history,
                "total_sec": total_sec, "config": CONFIG}, path)

print(f"START {SLUG}: epochs {start_epoch}..{CONFIG['epochs']}")
for epoch in range(start_epoch, CONFIG["epochs"] + 1):
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    tr_loss = train_one_epoch()
    val_m, _ = evaluate(model, val_loader, desc="  val")
    lr_now = optimizer.param_groups[0]["lr"]
    scheduler.step()
    dt = time.time() - t0; total_sec += dt
    mem = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0
    row = {"epoch": epoch, "train_loss": round(tr_loss, 5), "lr": lr_now,
           "epoch_time_sec": round(dt, 2), "gpu_mem_peak_gb": round(mem, 3)}
    row.update({f"val_{k}": round(v, 5) for k, v in val_m.items()})
    history.append(row)
    with open(csv_path, "a", newline="") as f:
        csv.writer(f).writerow([row[k] for k in CSV_HEADER])
    flag = ""
    if val_m["dice"] > best_val:
        best_val, best_epoch = val_m["dice"], epoch
        torch.save(model.state_dict(), os.path.join(ckpt_dir, "best.pth")); flag = "  <- best"
    save_ckpt(os.path.join(ckpt_dir, "last.pth"), epoch)
    print(f"Ep {epoch:3d} | loss {tr_loss:.4f} | val Dice {val_m['dice']:.4f} IoU {val_m['iou']:.4f} | "
          f"lr {lr_now:.2e} | {dt:5.1f}s{flag}")

print(f"DONE. best val Dice {best_val:.4f} @ epoch {best_epoch} | {total_sec/60:.1f} min")

In [ ]:
# ============================================================
# Cell 12 — Training curves
# ============================================================
log = pd.read_csv(csv_path)
fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
ax[0].plot(log.epoch, log.train_loss); ax[0].set_title("train loss (rate 1.0)")
ax[1].plot(log.epoch, log.val_dice, color="green"); ax[1].axhline(best_val, ls="--", color="green", alpha=.5)
ax[1].set_title(f"val Dice (best {best_val:.4f} @ {best_epoch})")
ax[2].plot(log.epoch, log.lr, color="k"); ax[2].set_yscale("log"); ax[2].set_title("LR")
ax[3].plot(log.epoch, log.epoch_time_sec, color="teal"); ax[3].set_title("epoch time (s)")
for a in ax: a.set_xlabel("epoch"); a.grid(alpha=.3)
plt.suptitle(SLUG); plt.tight_layout()
plt.savefig(os.path.join(out_dir, "training_curves.png"), dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# ============================================================
# Cell 13 — Test (ONCE): best-val checkpoint + last epoch, per-image CSV, results record
# ============================================================
model.load_state_dict(torch.load(os.path.join(ckpt_dir, "best.pth"), map_location=DEVICE))
test_best, per_img   = evaluate(model, test_loader, "official",  desc="test best")
test_best_corr, _    = evaluate(model, test_loader, "corrected", desc="test best (H,W)")
per_img.to_csv(os.path.join(out_dir, "per_image_test.csv"), index=False)

last = torch.load(os.path.join(ckpt_dir, "last.pth"), map_location=DEVICE, weights_only=False)
model.load_state_dict(last["model"])
test_last, per_img_last = evaluate(model, test_loader, "official", desc="test last")
per_img_last.to_csv(os.path.join(out_dir, "per_image_test_last.csv"), index=False)

results = {
    "slug": SLUG, "experiment": EXPERIMENT, "variant": VARIANT, "run_number": RUN_NUMBER, "seed": SEED,
    "encoder": CONFIG["encoder"], "multiscale_mode": CONFIG["multiscale_mode"], "use_amp": CONFIG["use_amp"],
    "params_total": sum(p.numel() for p in model.parameters()),
    "params_lgag": sum(p.numel() for n, p in model.named_parameters() if ".lgag" in n),
    "best_epoch": best_epoch, "best_val_dice": best_val,
    "test_best_official": test_best, "test_best_corrected_orientation": test_best_corr,
    "test_last_official": test_last,
    "train_minutes": round(total_sec / 60, 1),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
}
with open(os.path.join(out_dir, "results_record.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"{'':28s}{'Dice':>8s}{'IoU':>8s}")
for name, m in [("test @ best val (official)", test_best), ("test @ best val (H,W fixed)", test_best_corr),
                ("test @ last epoch (official)", test_last)]:
    print(f"{name:28s}{m['dice']:8.4f}{m['iou']:8.4f}")
print(f"best val Dice {best_val:.4f} @ epoch {best_epoch} | train {total_sec/60:.1f} min")

In [ ]:
# ============================================================
# Cell 14 — Prediction grid + output inventory
# ============================================================
model.load_state_dict(torch.load(os.path.join(ckpt_dir, "best.pth"), map_location=DEVICE)); model.eval()
ds = test_loader.dataset
mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
fig, ax = plt.subplots(4, 3, figsize=(9, 12))
with torch.no_grad():
    for r, idx in enumerate([0, len(ds)//3, 2*len(ds)//3, len(ds)-1]):
        img, gt, _, name = ds[idx]
        p = model(img[None].to(DEVICE))[-1].sigmoid()[0, 0].cpu().numpy()
        ax[r, 0].imshow((img.permute(1, 2, 0).numpy() * std + mean).clip(0, 1)); ax[r, 0].set_title(name, fontsize=8)
        ax[r, 1].imshow(gt[0], cmap="gray"); ax[r, 1].set_title("GT", fontsize=8)
        ax[r, 2].imshow(p > 0.5, cmap="gray"); ax[r, 2].set_title("pred", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(out_dir, "test_predictions.png"), dpi=120, bbox_inches="tight"); plt.show()

for p in sorted(glob.glob(os.path.join(out_dir, "*")) + glob.glob(os.path.join(ckpt_dir, "*"))):
    print(f"  {os.path.getsize(p)/1e6:8.2f} MB  {p}")
print("Save Version -> Quick Save to keep these files.")